# **Add partitioning results to dataset**

**Author**: Lukas Hörtnagl (holukas@ethz.ch)

# Info

- Previously gap-filled TA, SW_IN (Rg) and VPD were used to partition NEE (daytime and nighttime method)
- Using NEE fluxes gap-filled with random forest
- Using the partitioning algorithm implementations in `REddyProc`

# Imports

In [4]:
from pathlib import Path
import pandas as pd
from diive.core.io.files import save_parquet, load_parquet
from diive.core.times.times import TimestampSanitizer

# Load partitioning results for fluxes gap-filled with random forest

In [6]:
# partitioning_results = "81.1_CH-CHA_NEE_RF-GAPF_PART_RP-20250319215835.csv"
partitioning_results = "del.csv"
results = pd.read_csv(partitioning_results)
results = results.set_index("TIMESTAMP")
results.index.name = "TIMESTAMP_END"
results = TimestampSanitizer(data=results).get()
results

,Tair_orig,Tair_f,Tair_fqc,Tair_fall,Tair_fall_qc,Tair_fnum,Tair_fsd,Tair_fmeth,Tair_fwin,Rg_orig,Rg_f,Rg_fqc,Rg_fall,Rg_fall_qc,Rg_fnum,...,FP_GPP2000,FP_k,FP_beta,FP_alpha,FP_RRef,FP_E0,FP_k_sd,FP_beta_sd,FP_alpha_sd,FP_RRef_sd,FP_E0_sd,Reco_DT_U50,GPP_DT_U50,Reco_DT_U50_SD,GPP_DT_U50_SD
TIMESTAMP_MIDDLE,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2005-01-01 00:15:00,1.566667,1.566667,0,1.566667,NaN,NaN,NaN,NaN,NaN,0,0,0,0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.093071,0,0.080016,0
2005-01-01 00:45:00,1.533333,1.533333,0,1.533333,NaN,NaN,NaN,NaN,NaN,0,0,0,0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.092682,0,0.079688,0
2005-01-01 01:15:00,1.566667,1.566667,0,1.566667,NaN,NaN,NaN,NaN,NaN,0,0,0,0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.093071,0,0.080016,0
2005-01-01 01:45:00,1.566667,1.566667,0,1.566667,NaN,NaN,NaN,NaN,NaN,0,0,0,0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.093071,0,0.080016,0
2005-01-01 02:15:00,1.500000,1.500000,0,1.500000,NaN,NaN,NaN,NaN,NaN,0,0,0,0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.092295,0,0.079361,0


# Identify GPP and RECO columns

In [18]:
partcols = [c for c in results.columns if any(substring in c for substring in ["GPP", "Reco"])];
partcols = [c for c in partcols if not str(c).endswith("_fqc")]  # These are from REddyProc's MDS gap-filling, but data were already gap-filled, therefore not needed
partcols

['Reco_U16',
 'GPP_U16_f',
 'Reco_DT_U16',
 'GPP_DT_U16',
 'Reco_DT_U16_SD',
 'GPP_DT_U16_SD',
 'Reco_U84',
 'GPP_U84_f',
 'Reco_DT_U84',
 'GPP_DT_U84',
 'Reco_DT_U84_SD',
 'GPP_DT_U84_SD',
 'Reco_U50',
 'GPP_U50_f',
 'FP_GPP2000',
 'Reco_DT_U50',
 'GPP_DT_U50',
 'Reco_DT_U50_SD',
 'GPP_DT_U50_SD']

# Create subset with GPP and RECO columns

In [19]:
subset_partcols = results[partcols].copy()
subset_partcols

,Reco_U16,GPP_U16_f,Reco_DT_U16,GPP_DT_U16,Reco_DT_U16_SD,GPP_DT_U16_SD,Reco_U84,GPP_U84_f,Reco_DT_U84,GPP_DT_U84,Reco_DT_U84_SD,GPP_DT_U84_SD,Reco_U50,GPP_U50_f,FP_GPP2000,Reco_DT_U50,GPP_DT_U50,Reco_DT_U50_SD,GPP_DT_U50_SD
TIMESTAMP_MIDDLE,,,,,,,,,,,,,,,,,,,
2005-01-01 00:15:00,1.800748,0.562911,0.476922,0,0.293865,0,1.746895,1.105825,0.089205,0,0.122912,0,1.830543,0.918553,NaN,0.093071,0,0.080016,0
2005-01-01 00:45:00,1.799303,0.575336,0.475155,0,0.292817,0,1.744107,1.101436,0.088843,0,0.122413,0,1.828898,0.917972,NaN,0.092682,0,0.079688,0
2005-01-01 01:15:00,1.800748,0.170341,0.476922,0,0.293865,0,1.746895,0.462104,0.089205,0,0.122912,0,1.830543,0.163001,NaN,0.093071,0,0.080016,0
2005-01-01 01:45:00,1.800748,0.277298,0.476922,0,0.293865,0,1.746895,0.460866,0.089205,0,0.122912,0,1.830543,0.190890,NaN,0.093071,0,0.080016,0
2005-01-01 02:15:00,1.797856,0.189333,0.473392,0,0.291772,0,1.741320,0.402870,0.088482,0,0.121916,0,1.827253,0.167042,NaN,0.092295,0,0.079361,0


# Rename partitioning variables

These original NEE flux columns were renamed and then used during partitioning:
- NEE_L3.1_L3.3_CUT_16_QCF_gfRF
- NEE_L3.1_L3.3_CUT_50_QCF_gfRF
- NEE_L3.1_L3.3_CUT_84_QCF_gfRF

In [21]:
# NEE_L3.1_L3.3_CUT_16_QCF_gfRF

renaming_dict = {
    'FP_GPP2000': 'FP_GPP2000',

    'GPP_DT_U16': 'GPP_DT_CUT_16_gfRF',
    'GPP_DT_U16_SD': 'GPP_DT_CUT_16_gfRF_SD',
    'GPP_DT_U50': 'GPP_DT_CUT_50_gfRF',
    'GPP_DT_U50_SD': 'GPP_DT_CUT_50_gfRF_SD',
    'GPP_DT_U84': 'GPP_DT_CUT_84_gfRF',
    'GPP_DT_U84_SD': 'GPP_DT_CUT_84_gfRF_SD',

    'GPP_U16_f': 'GPP_NT_CUT_16_gfRF',    
    'GPP_U50_f': 'GPP_NT_CUT_50_gfRF',    
    'GPP_U84_f': 'GPP_NT_CUT_84_gfRF',    

    'Reco_DT_U16': 'RECO_DT_CUT_16_gfRF',
    'Reco_DT_U16_SD': 'RECO_DT_CUT_16_gfRF_SD',
    'Reco_DT_U50': 'RECO_DT_CUT_50_gfRF',
    'Reco_DT_U50_SD': 'RECO_DT_CUT_50_gfRF_SD',
    'Reco_DT_U84': 'RECO_DT_CUT_84_gfRF',
    'Reco_DT_U84_SD': 'RECO_DT_CUT_84_gfRF_SD',

    'Reco_U16': 'RECO_NT_CUT_16_gfRF',
    'Reco_U50': 'RECO_NT_CUT_50_gfRF',
    'Reco_U84': 'RECO_NT_CUT_84_gfRF',
}
subset_partcols = subset_partcols.rename(columns=renaming_dict, inplace=False)
subset_partcols

,RECO_NT_CUT_16_gfRF,GPP_NT_CUT_16_gfRF,RECO_DT_CUT_16_gfRF,GPP_DT_CUT_16_gfRF,RECO_DT_CUT_16_gfRF_SD,GPP_DT_CUT_16_gfRF_SD,RECO_NT_CUT_84_gfRF,GPP_NT_CUT_84_gfRF,RECO_DT_CUT_84_gfRF,GPP_DT_CUT_84_gfRF,RECO_DT_CUT_84_gfRF_SD,GPP_DT_CUT_84_gfRF_SD,RECO_NT_CUT_50_gfRF,GPP_NT_CUT_50_gfRF,FP_GPP2000,RECO_DT_CUT_50_gfRF,GPP_DT_CUT_50_gfRF,RECO_DT_CUT_50_gfRF_SD,GPP_DT_CUT_50_gfRF_SD
TIMESTAMP_MIDDLE,,,,,,,,,,,,,,,,,,,
2005-01-01 00:15:00,1.800748,0.562911,0.476922,0,0.293865,0,1.746895,1.105825,0.089205,0,0.122912,0,1.830543,0.918553,NaN,0.093071,0,0.080016,0
2005-01-01 00:45:00,1.799303,0.575336,0.475155,0,0.292817,0,1.744107,1.101436,0.088843,0,0.122413,0,1.828898,0.917972,NaN,0.092682,0,0.079688,0
2005-01-01 01:15:00,1.800748,0.170341,0.476922,0,0.293865,0,1.746895,0.462104,0.089205,0,0.122912,0,1.830543,0.163001,NaN,0.093071,0,0.080016,0
2005-01-01 01:45:00,1.800748,0.277298,0.476922,0,0.293865,0,1.746895,0.460866,0.089205,0,0.122912,0,1.830543,0.190890,NaN,0.093071,0,0.080016,0
2005-01-01 02:15:00,1.797856,0.189333,0.473392,0,0.291772,0,1.741320,0.402870,0.088482,0,0.121916,0,1.827253,0.167042,NaN,0.092295,0,0.079361,0


# Load main data

In [ ]:
SOURCEDIR = r"../30_MERGE_DATA"
FILENAME = r"33.5_CH-CHA_IRGA+QCL+LGR+M10+MGMT_Level-1_eddypro_fluxnet_2005-2024.parquet"
FILEPATH = Path(SOURCEDIR) / FILENAME
maindf = load_parquet(filepath=FILEPATH)

## Add results for NEE

In [ ]:
newcols = [c for c in nee.columns if c not in maindf]
print("NEW VARIABLES FROM FLUX PROCESSING CHAIN:")
[print(f"++ {c}") for c in newcols]
maindf = pd.concat([maindf, nee[newcols]], axis=1)
maindf

## Add results for LE

In [ ]:
newcols = [c for c in le.columns if c not in maindf]
print("NEW VARIABLES FROM FLUX PROCESSING CHAIN:")
[print(f"++ {c}") for c in newcols]
maindf = pd.concat([maindf, le[newcols]], axis=1)
maindf

## Add results for H

In [ ]:
newcols = [c for c in h.columns if c not in maindf]
print("NEW VARIABLES FROM FLUX PROCESSING CHAIN:")
[print(f"++ {c}") for c in newcols]
maindf = pd.concat([maindf, h[newcols]], axis=1)
maindf

## Add results for FN2O

In [ ]:
newcols = [c for c in n2o.columns if c not in maindf]
print("NEW VARIABLES FROM FLUX PROCESSING CHAIN:")
[print(f"++ {c}") for c in newcols]
maindf = pd.concat([maindf, n2o[newcols]], axis=1)
maindf

## Add results for CH4

In [ ]:
newcols = [c for c in ch4.columns if c not in maindf]
print("NEW VARIABLES FROM FLUX PROCESSING CHAIN:")
[print(f"++ {c}") for c in newcols]
maindf = pd.concat([maindf, ch4[newcols]], axis=1)
maindf

# Export 

## Export all data

In [ ]:
filename = "61.1_FLUXES_M10_MGMT_L4.1_NEE_LE_H_FN2O_FCH4"
maindf.to_csv(f"{filename}.csv", index=True)
save_parquet(data=maindf, filename=filename)

# **End of notebook**

Congratulations, you reached the end of this notebook! Before you go let's store your finish time.

In [ ]:
from datetime import datetime
dt_string = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Finished. {dt_string}")